In [ ]:
pip install -r requirements.txt

In [2]:
import numpy as np
import cv2
import torch
from torchvision import transforms
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from file_loader import get_all_files

from data_utils import PreClassificationDataset, compute_class_weights
from network.classification_net import PreClassificationModel
from solver import Solver

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

In [3]:
data = 'data/images/'                             # labelled images, which are to classified as good/bad
files = get_all_files(data, 'png')
len(files)

13648

In [4]:
file_test = open("test.txt", "w")  

for file in files:
    file_test.write( file + "\n")  

file_test.close()

In [5]:
import torch.utils.data as data
class PreClassificationDataset_test(data.Dataset):
    
    def __init__(self, image_paths_file, to_tensor_center, to_tensor_full):
        with open(image_paths_file) as f:
            self.image_names = f.read().splitlines()
        self.to_tensor_center = to_tensor_center
        self.to_tensor_full = to_tensor_full
     
                                             
    def __getitem__(self, key):
        if isinstance(key, slice):
            # get the start, stop, and step from the slice
            return [self[ii] for ii in range(*key.indices(len(self)))]
        elif isinstance(key, int):
            # handle negative indices
            if key < 0:
                key += len(self)
            if key < 0 or key >= len(self):
                raise IndexError("The index (%d) is out of range." % key)
            # get the data from direct index
            return self.get_item_from_index(key)
        else:
            raise TypeError("Invalid argument type.")

    def __len__(self):
        return len(self.image_names)

    def get_item_from_index(self, index):
        
        image_path = self.image_names[index]
        
        img_full = cv2.imread(image_path)   
        img_full = cv2.cvtColor(img_full, cv2.COLOR_BGR2RGB) ## opencv reads the color channels in reverse order :(
      
        
        image_path_center = image_path.replace('images','test_center_cell_images_latest')
        
        img_center = cv2.imread(image_path_center)   
        img_center = cv2.cvtColor(img_center, cv2.COLOR_BGR2RGB)
                     
    
        
        img_center = self.to_tensor_center(img_center) 
        img_full = self.to_tensor_full(img_full)
        img = torch.cat((img_center, img_full), 0)     # concatenate tensors along channels
     
        return img
    
to_tensor_testset_center = transforms.Compose([transforms.ToTensor(),
                               transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))])         
                               
to_tensor_testset_full = transforms.Compose([transforms.ToTensor(),
                               transforms.Normalize((0.6846, 0.6206, 0.6555), (0.0833, 0.1287, 0.0752))])        

test_data = PreClassificationDataset_test(image_paths_file = 'test.txt',\
            to_tensor_center = to_tensor_testset_center, to_tensor_full = to_tensor_testset_full)

print("Test size: %i" % len(test_data))
print("Concatenated Img size: ", test_data[0].size())

Test size: 13648
Concatenated Img size:  torch.Size([6, 120, 120])


### Testing model on labelled data

In [6]:
model = torch.load("models/preclassification_epoch_30.model")
model.to(device)

from torch.autograd import Variable
test_dataloader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False, num_workers=1)
predictions = []
model.eval()
for inputs in test_dataloader:
                
    inputs = Variable(inputs)
                
    inputs = inputs.to(device)
    outputs = model(inputs)
    
    pred = np.array(outputs.cpu() > 0.5, dtype=float)
    predictions.append(pred)
    
model.train()
predictions = np.squeeze(np.array(predictions))
predictions.shape

(13648,)

### Getting good images from the prediction label

In [7]:
good_image_names =[]
import os
import shutil
good_image_names = [files[i] for i in list(np.where(predictions==0)[0])]           #good images for classification
for img_name in good_image_names:
    file = img_name.replace('images','good_images')                   # getting full crop good image
    out_dir, _ =os.path.split(file)
    if not os.path.isdir(out_dir):
        os.makedirs(out_dir)
    shutil.copy(img_name,out_dir)
    
    img_name = img_name.replace('images','test_center_cell_images_latest')
    file = file.replace('good_images','good_images_center')     # getting center cell good image
    out_dir, _ =os.path.split(file)
    if not os.path.isdir(out_dir):
        os.makedirs(out_dir)
    shutil.copy(img_name,out_dir)
    

In [8]:
data = 'data/good_images/'
files_ = get_all_files(data, 'png')
len(files_)

12064